# 🎬 TUS Colab Pipeline — Wan 2.2 Video Generator

**TikTok UGC Studio** — Automatic video generation pipeline for Colab Pro

## Workflow:
1. TUS (Server) → Scrapes product + Generates TTS → Writes job to `_ready/`
2. **You click Play → This notebook reads `_ready/` → Runs Wan 2.2 I2V → Writes to `_done/`**
3. TUS (Server) → Detects `_done/` → Posts via PostForMe

---
### 📁 Required Google Drive Structure:
```
MyDrive/TUS_Pipeline/
  ├── _ready/        ← TUS writes jobs here
  ├── _done/         ← This notebook writes results here
  ├── _archive/      ← Archived jobs
  └── libs/          ← Pre-downloaded models (Wan 2.2, etc.)
```

---
## 📋 Cell 1: Mount Google Drive & Install Dependencies

In [ ]:
# ⚡ Cell 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted!')

In [ ]:
# ⚡ Cell 2: Install Dependencies (run once)
# Wan 2.2 runs via HuggingFace Diffusers
%pip install -q diffusers>=0.33.0 transformers accelerate torch torchvision torchaudio sentencepiece einops opencv-python pillow

# FFmpeg (for compose)
!apt-get -qq install -y ffmpeg > /dev/null 2>&1

print('✅ Dependencies installed!')

---
## ⚙️ Cell 3: Configure Pipeline

This sets up paths, loads Wan 2.2 model, and prepares the pipeline functions.

In [ ]:
# ⚡ Cell 3: Pipeline Configuration
import os
import json
import shutil
import time
import logging
import subprocess
from pathlib import Path
from datetime import datetime
from typing import Optional

logging.basicConfig(level=logging.INFO, format='%(asctime)s [%(levelname)s] %(message)s')
logger = logging.getLogger('tus_colab')

# ─── Paths ─────────────────────────────────────────────────────────────────
DRIVE_ROOT = Path('/content/drive/MyDrive/TUS_Pipeline')
READY_DIR = DRIVE_ROOT / '_ready'
DONE_DIR = DRIVE_ROOT / '_done'
LIBS_DIR = DRIVE_ROOT / 'libs'

DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
READY_DIR.mkdir(exist_ok=True)
DONE_DIR.mkdir(exist_ok=True)
LIBS_DIR.mkdir(exist_ok=True)

logger.info(f'📂 Drive Root: {DRIVE_ROOT}')
logger.info(f'📂 Ready:     {READY_DIR}')
logger.info(f'📂 Done:      {DONE_DIR}')
logger.info(f'📂 Libs:      {LIBS_DIR}')

In [ ]:
# ⚡ Cell 4: Load Wan 2.2 Model
# This downloads ~10GB on first run, cached in Drive libs/ for reuse

import torch
from diffusers import WanImageToVideoPipeline

MODEL_ID = 'Wan-AI/Wan2.2-I2V-A14B-Diffusers'

# Check if model is already cached locally
cache_dir = str(LIBS_DIR / 'wan2.2_i2v')

logger.info(f'🔄 Loading Wan 2.2 I2V-A14B (first time may take 15-30 mins to download)...')
logger.info(f'   Model: {MODEL_ID}')
logger.info(f'   Cache: {cache_dir}')

# Load with bfloat16 for V100/A100 — saves VRAM
pipe = WanImageToVideoPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    cache_dir=cache_dir,
    variant='bf16',
)
pipe.to('cuda')

# Enable memory optimization (important for Colab!)
pipe.enable_model_cpu_offload()
pipe.vae.enable_slicing()
pipe.vae.enable_tiling()

logger.info(f'✅ Wan 2.2 model loaded! VRAM used: {torch.cuda.memory_allocated()/1024**3:.1f} GB')

---
## 🛠️ Cell 5: Pipeline Functions

In [ ]:
# ⚡ Cell 5: Core Pipeline Functions

def get_pending_jobs() -> list:
    """List all jobs in _ready/ sorted by creation time."""
    if not READY_DIR.exists():
        return []
    jobs = []
    for job_dir in sorted(READY_DIR.iterdir()):
        if not job_dir.is_dir():
            continue
        config_file = job_dir / 'config.json'
        if config_file.exists():
            with open(config_file, 'r') as f:
                config = json.load(f)
            config['_job_dir'] = str(job_dir)
            jobs.append(config)
    jobs.sort(key=lambda j: j.get('created_at', ''))
    return jobs


def run_wan_i2v(
    image_path: str,
    prompt: str,
    duration: int = 5,
    output_path: str = None,
) -> str:
    """Run Wan 2.2 Image-to-Video generation.
    
    Args:
        image_path: Path to product image (input)
        prompt: Text prompt describing the motion
        duration: Target video duration in seconds (Wan 2.2 max ~5s per gen)
        output_path: Output video file path
    
    Returns:
        Path to generated video
    """
    from PIL import Image
    
    if output_path is None:
        output_path = f'/tmp/wan_output_{int(time.time())}.mp4'
    
    # Load image
    input_image = Image.open(image_path).convert('RGB')
    logger.info(f'   ├─ Image: {image_path} ({input_image.size})')
    logger.info(f'   ├─ Prompt: {prompt[:60]}...')
    logger.info(f'   ├─ Duration: {duration}s')
    
    # Wan 2.2 max ~81 frames (5.1s at 16fps) or ~121 frames
    # We generate in chunks for longer videos
    num_frames = min(int(duration * 16), 81)  # 16fps, max 81 frames ≈ 5s
    
    # Generate
    t0 = time.time()
    with torch.autocast('cuda'):
        video_frames = pipe(
            image=input_image,
            prompt=prompt,
            num_frames=num_frames,
            num_inference_steps=25,
            guidance_scale=5.0,
            height=480,  # 480p for speed/VRAM balance
            width=848,   # 9:16 portrait mode (480x848 ≈ 9:16)
        ).frames[0]
    gen_time = time.time() - t0
    logger.info(f'   └─ Generated {len(video_frames)} frames in {gen_time:.1f}s ({len(video_frames)/gen_time:.1f} fps)')
    
    # Save as video using torchvision
    from torchvision.io import write_video
    import torch
    
    # Stack frames to tensor [T, H, W, C]
    frames_tensor = torch.stack([
        torch.tensor(frame) if not isinstance(frame, torch.Tensor) else frame
        for frame in video_frames
    ])  # [T, H, W, C]
    
    # Convert [0,1] float to [0,255] uint8
    if frames_tensor.dtype in (torch.float32, torch.bfloat16, torch.float16):
        frames_tensor = (frames_tensor * 255).clamp(0, 255).to(torch.uint8)
    
    # Ensure H, W are multiples of 2
    if frames_tensor.shape[1] % 2 != 0:
        frames_tensor = frames_tensor[:, :-1, :, :]
    if frames_tensor.shape[2] % 2 != 0:
        frames_tensor = frames_tensor[:, :, :-1, :]
    
    write_video(
        output_path,
        frames_tensor,
        fps=16,
        codec='libx264',
        video_codec='h264',
    )
    
    size_mb = os.path.getsize(output_path) / (1024 * 1024)
    logger.info(f'   ✅ Video saved: {output_path} ({size_mb:.1f} MB)')
    
    return output_path


def compose_video(video_path: str, audio_path: str, output_path: str):
    """Merge generated video with TTS audio using FFmpeg."""
    cmd = [
        'ffmpeg', '-y',
        '-i', video_path,
        '-i', audio_path,
        '-c:v', 'copy',
        '-c:a', 'aac',
        '-map', '0:v:0',
        '-map', '1:a:0',
        '-shortest',
        output_path,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'FFmpeg compose failed: {result.stderr}')
    logger.info(f'   ✅ Composed: {output_path}')


def mark_complete(job_id: str, video_path: str, metrics: dict = None):
    """Mark a job as complete — move to _done/."""
    ready_job = READY_DIR / job_id
    done_job = DONE_DIR / job_id
    done_job.mkdir(parents=True, exist_ok=True)

    result = {
        'job_id': job_id,
        'status': 'completed',
        'completed_at': datetime.now().isoformat(),
        'video_file': os.path.basename(video_path),
        'video_size_bytes': os.path.getsize(video_path),
        'metrics': metrics or {},
    }

    with open(done_job / 'result.json', 'w') as f:
        json.dump(result, f, indent=2)

    shutil.copy2(video_path, done_job / 'final_output.mp4')

    # Clean up _ready/
    if ready_job.exists():
        shutil.rmtree(str(ready_job))

    logger.info(f'✅ Job {job_id} complete → {done_job}')
    return result


def mark_failed(job_id: str, error: str):
    """Mark a job as failed."""
    ready_job = READY_DIR / job_id
    done_job = DONE_DIR / job_id
    done_job.mkdir(parents=True, exist_ok=True)

    result = {
        'job_id': job_id,
        'status': 'failed',
        'completed_at': datetime.now().isoformat(),
        'error': str(error),
    }

    with open(done_job / 'result.json', 'w') as f:
        json.dump(result, f, indent=2)

    if ready_job.exists():
        shutil.rmtree(str(ready_job))

    logger.error(f'❌ Job {job_id} failed: {error}')


def process_job(config: dict) -> dict:
    """Process a single job end-to-end.
    
    1. Load product image + TTS audio
    2. Run Wan 2.2 I2V
    3. Compose video + audio
    4. Write result to _done/
    """
    job_id = config['job_id']
    job_dir = Path(config['_job_dir'])
    
    logger.info(f'=' * 60)
    logger.info(f'🎬 Processing job: {job_id}')
    logger.info(f'   Product: {config.get("product_title", "")[:60]}')
    logger.info(f'   Duration: {config.get("duration", 5)}s')
    logger.info(f'   Platforms: {config.get("platforms", ["tiktok"])}')
    logger.info(f'=' * 60)
    
    # Paths
    image_path = str(job_dir / config['product_image'])
    audio_path = str(job_dir / config['tts_audio'])
    prompt = config.get('hook', '') or config.get('product_title', '')
    duration = config.get('duration', 5)
    
    # Validate assets
    if not os.path.exists(image_path):
        raise FileNotFoundError(f'Product image not found: {image_path}')
    if not os.path.exists(audio_path):
        raise FileNotFoundError(f'TTS audio not found: {audio_path}')
    
    t_start = time.time()
    
    # Step 1: Wan I2V
    logger.info(f'[Step 1/2] 🎬 Generating video from product image...')
    gen_video = run_wan_i2v(
        image_path=image_path,
        prompt=f'Product showcase: {prompt}. Smooth camera motion, product rotates slowly.',
        duration=min(duration, 5),  # Wan 2.2 max ~5s per gen
    )
    
    # Step 2: Compose with TTS audio
    logger.info(f'[Step 2/2] 🎞️ Composing final video with TTS audio...')
    output_path = f'/tmp/{job_id}_final.mp4'
    compose_video(gen_video, audio_path, output_path)
    
    total_time = time.time() - t_start
    
    metrics = {
        'total_time_seconds': round(total_time, 1),
        'video_size_mb': round(os.path.getsize(output_path) / (1024 * 1024), 1),
        'duration_seconds': duration,
    }
    
    # Mark complete
    mark_complete(job_id, output_path, metrics)
    
    # Cleanup temp files
    for f in [gen_video, output_path]:
        if os.path.exists(f):
            os.remove(f)
    
    return metrics


logger.info('✅ Pipeline functions loaded!')

---
## ▶️ Cell 6: Process All Pending Jobs

**This is the main run cell.** Click Play to process all jobs in `_ready/`.

The notebook will:
1. Scan `_ready/` for jobs
2. Process each job one by one
3. Write results to `_done/`
4. TUS server will detect and auto-post via PostForMe

In [ ]:
# ⚡ Cell 6: ▶️ RUN ALL PENDING JOBS
# Click ▶️ to process all jobs waiting in _ready/

pending = get_pending_jobs()

if not pending:
    print('📭 No pending jobs in _ready/.')
    print('   1. Submit a job via TUS API: POST /colab/submit')
    print('   2. Or ask PeteAI: "สร้างคลิปสินค้า URL ..."')
else:
    total = len(pending)
    success = 0
    failed = 0
    
    print(f'📦 Found {total} job(s) to process!')
    print('=' * 60)
    
    for i, config in enumerate(pending, 1):
        title = config.get('product_title', '?')[:40]
        print(f'\n[{i}/{total}] Job: {config["job_id"]} — {title}')
        
        try:
            metrics = process_job(config)
            success += 1
            print(f'   ✅ Done! ({metrics["total_time_seconds"]}s, {metrics["video_size_mb"]}MB)')
        except Exception as e:
            mark_failed(config['job_id'], str(e))
            failed += 1
            print(f'   ❌ Failed: {e}')
    
    print('=' * 60)
    print(f'📊 Summary: {success} ✅ | {failed} ❌ | {total} total')
    
    if success > 0:
        print(f'\n✅ TUS will auto-detect _done/ and post via PostForMe!')
        print(f'   Check status: GET /colab/jobs/completed')

---
## 📊 Cell 7: Show Jobs Status

In [ ]:
# ⚡ Cell 7: Show Status
from datetime import datetime

pending = get_pending_jobs()
done_jobs = list(DONE_DIR.iterdir()) if DONE_DIR.exists() else []

print('📊 Pipeline Status')
print('=' * 50)
print(f'📥 _ready/:  {len(pending)} jobs')
print(f'📤 _done/:   {len(done_jobs)} jobs')
print()

if pending:
    print('📋 Pending Jobs:')
    for j in pending:
        created = j.get('created_at', '?')[:19]
        print(f'   🔄 {j["job_id"]} — {j.get("product_title", "")[:40]} ({created})')
    print()

if done_jobs:
    print('📋 Completed Jobs:')
    for d in sorted(done_jobs):
        result_file = d / 'result.json'
        if result_file.exists():
            with open(result_file) as f:
                r = json.load(f)
            status_icon = '✅' if r.get('status') == 'completed' else '❌'
            video = r.get('video_file', '?')
            print(f'   {status_icon} {d.name} — {video}')
        else:
            print(f'   ⏳ {d.name} — processing...')

---
## 🧪 Cell 8: Test — Generate Sample Video

Use this cell to generate a sample video for testing, without needing a TUS job.

In [ ]:
# ⚡ Cell 8: 🧪 Generate Sample Video (for testing)
# This creates a sample job locally for testing the pipeline

test_mode = input('Create a test sample? (y/n): ') if False else 'y'

if test_mode == 'y' or True:
    # Create a quick test with a random image
    from PIL import Image, ImageDraw, ImageFont
    
    print('🎨 Creating sample product image...')
    img = Image.new('RGB', (480, 848), color=(30, 30, 50))
    draw = ImageDraw.Draw(img)
    draw.rectangle([100, 200, 380, 500], fill=(200, 100, 50), outline=(255, 200, 100), width=3)
    draw.text((240, 600), 'TEST PRODUCT', fill='white', anchor='mt')
    
    sample_img = '/content/sample_product.png'
    img.save(sample_img)
    print(f'   └─ Image: {sample_img}')
    
    print('🎤 Generating sample TTS...')
    from gtts import gTTS
    tts = gTTS(text='สวัสดีครับ วันนี้เรามาดูสินค้าดีๆ กันครับ', lang='th', slow=False)
    sample_audio = '/content/sample_tts.mp3'
    tts.save(sample_audio)
    print(f'   └─ Audio: {sample_audio}')
    
    # Run pipeline
    print('🎬 Running Wan 2.2 I2V...')
    video = run_wan_i2v(
        image_path=sample_img,
        prompt='A product showcase with smooth camera panning, elegant lighting',
        duration=4,
    )
    
    print('🎞️ Composing with audio...')
    output = '/content/sample_final.mp4'
    compose_video(video, sample_audio, output)
    
    print(f'\n✅ Sample video: {output}')
    print(f'   Size: {os.path.getsize(output)/1024/1024:.1f} MB')
    from IPython.display import Video
    Video(output, width=360, height=640)

---
## ⏩ Bonus: One-Click Run

After first setup (Cells 1-4), run all jobs with one click from here.

In [ ]:
# ⏩ One-Click: Process ALL pending jobs
# Run this after the first time setup is complete

pending = get_pending_jobs()
total = len(pending)

if total == 0:
    print('📭 No jobs in _ready/')
else:
    print(f'🔄 Processing {total} job(s)...')
    for i, config in enumerate(pending, 1):
        try:
            m = process_job(config)
            print(f'  [{i}/{total}] ✅ {config["job_id"]} ({m["total_time_seconds"]}s)')
        except Exception as e:
            mark_failed(config['job_id'], str(e))
            print(f'  [{i}/{total}] ❌ {config["job_id"]}: {e}')
    print(f'\n✅ Done! TUS will auto-post via PostForMe.')